In [6]:
!pip install datasets pandas matplotlib seaborn scikit-learn Sastrawi nltk


In [10]:
import os
import random
import re
import kagglehub
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

random.seed(42)
np.random.seed(42)

# ==============================================================================
# 1. PINDAI & PILIH FILE YANG MEMILIKI 6 KOLOM ASPEK (dataset_labeled_v6.xlsx)
# ==============================================================================
dataset_path = kagglehub.dataset_download(
    "jareedhnnn/shopee-fashion-reviews-indonesian"
)

target_keywords = [
    "kualitas",
    "bahan",
    "ukuran",
    "desain",
    "warna",
    "kenyamanan",
]


def baca_file_data(path):
  ext = os.path.splitext(path)[1].lower()
  if ext == ".xlsx":
    return pd.read_excel(path)
  elif ext == ".csv":
    return pd.read_csv(path)
  elif ext == ".parquet":
    return pd.read_parquet(path)
  return None


kandidat_berlabel = []
for root, _, files in os.walk(dataset_path):
  for f in files:
    full_path = os.path.join(root, f)
    temp_df = baca_file_data(full_path)
    if temp_df is None or len(temp_df) == 0:
      continue

    cols_lower = [c.lower() for c in temp_df.columns]
    # Cek apakah file memiliki semua 6 kata kunci aspek (dan bukan awalan 'gold_')
    punya_6_aspek = all(
        any(kw in c and not c.startswith("gold_") for c in cols_lower)
        for kw in target_keywords
    )
    if punya_6_aspek:
      kandidat_berlabel.append((f, temp_df))

if not kandidat_berlabel:
  raise RuntimeError(
      "Tidak ditemukan file dengan 6 kolom aspek di dalam direktori Kaggle."
  )

# Ambil file berlabel 6 aspek dengan baris terbanyak (dataset_labeled_v6.xlsx)
kandidat_berlabel.sort(key=lambda x: len(x[1]), reverse=True)
selected_file, df = kandidat_berlabel[0]

print(f"-> File utama terpilih : '{selected_file}' ({len(df)} baris)")
print(f"-> Kolom tersedia      : {list(df.columns)}")

# ==============================================================================
# 2. IDENTIFIKASI KOLOM TEKS & 6 KOLOM ASPEK
# ==============================================================================
ASPECT_COLS = []
for kw in target_keywords:
  cocok = [
      c
      for c in df.columns
      if kw in c.lower() and not c.lower().startswith("gold_")
  ]
  ASPECT_COLS.append(cocok[0])

kandidat_teks = ["ulasan", "review", "comment", "text", "content", "komentar"]
text_col = next(
    (c for c in df.columns if c.lower() in kandidat_teks), None
)
if text_col is None:
  sisa_kolom = [
      c
      for c in df.columns
      if c not in ASPECT_COLS and df[c].dtype == "object"
  ]
  text_col = sisa_kolom[0]

print(f"-> Kolom teks          : '{text_col}'")
print(f"-> 6 Kolom aspek       : {ASPECT_COLS}")

# ==============================================================================
# 3. DATA CLEANING & CEK JUMLAH BARIS PER TAHAP
# ==============================================================================
df_bersih = df.dropna(subset=[text_col] + ASPECT_COLS).copy()
n_dropna = len(df_bersih)

df_bersih = df_bersih.drop_duplicates(subset=[text_col]).copy()
n_dedup = len(df_bersih)

df_bersih["char_length"] = df_bersih[text_col].astype(str).str.len()
df_bersih = df_bersih[df_bersih["char_length"] >= 5].drop(
    columns=["char_length"]
)
n_len5 = len(df_bersih)


def bersihkan_teks(teks):
  teks = str(teks).lower()
  teks = re.sub(r"[^a-z\s]", " ", teks)
  return re.sub(r"\s+", " ", teks).strip()


df_bersih["review_clean"] = df_bersih[text_col].apply(bersihkan_teks)
df_bersih = df_bersih[df_bersih["review_clean"] != ""].reset_index(drop=True)

# ==============================================================================
# 4. ENCODING 6 ASPEK & DATA SPLITTING (80:10:10)
# ==============================================================================
label_mapping = {"negatif": 0, "netral": 1, "positif": 2}
encoded_cols = []

for col in ASPECT_COLS:
  clean_label = df_bersih[col].astype(str).str.lower().str.strip()
  enc_col = f"{col}_encoded"
  df_bersih[enc_col] = clean_label.map(label_mapping)
  if df_bersih[enc_col].isna().any():
    raise ValueError(
        f"Label tidak dikenal pada '{col}': {clean_label.unique()}"
    )
  df_bersih[enc_col] = df_bersih[enc_col].astype(int)
  encoded_cols.append(enc_col)

X = df_bersih["review_clean"]
Y = df_bersih[encoded_cols]
minority_col = f"{ASPECT_COLS[3]}_encoded"  # aspek_desain

try:
  strat_key = Y.astype(str).agg("_".join, axis=1)
  X_temp, X_test, Y_temp, Y_test = train_test_split(
      X, Y, test_size=0.10, random_state=42, stratify=strat_key
  )
  strat_temp = Y_temp.astype(str).agg("_".join, axis=1)
  X_train, X_val, Y_train, Y_val = train_test_split(
      X_temp, Y_temp, test_size=0.1111, random_state=42, stratify=strat_temp
  )
except ValueError:
  X_temp, X_test, Y_temp, Y_test = train_test_split(
      X, Y, test_size=0.10, random_state=42, stratify=Y[minority_col]
  )
  X_train, X_val, Y_train, Y_val = train_test_split(
      X_temp,
      Y_temp,
      test_size=0.1111,
      random_state=42,
      stratify=Y_temp[minority_col],
  )

# ==============================================================================
# 5. TAMPILKAN TABEL RINGKASAN JUMLAH
# ==============================================================================
print("\n" + "=" * 68)
print("1. RINGKASAN JUMLAH BARIS DATA")
print("=" * 68)
ringkasan = pd.DataFrame({
    "Tahap / Himpunan": [
        "Dataset Awal (dataset_labeled_v6.xlsx)",
        "Setelah dropna()",
        "Setelah hapus duplikat teks",
        "Setelah filter >= 5 karakter",
        "Final Data Bersih (df_bersih)",
        "Data Training (80%)",
        "Data Validation (10%)",
        "Data Testing (10%)",
    ],
    "Jumlah Baris": [
        len(df),
        n_dropna,
        n_dedup,
        n_len5,
        len(df_bersih),
        len(X_train),
        len(X_val),
        len(X_test),
    ],
})
print(ringkasan.to_string(index=False))

print("\n" + "=" * 68)
print("2. DISTRIBUSI SENTIMEN KE-6 ASPEK (SETELAH CLEANING)")
print("=" * 68)
rekap = []
for col in encoded_cols:
  counts = df_bersih[col].value_counts().reindex([0, 1, 2], fill_value=0)
  tot = counts.sum()
  rekap.append({
      "Aspek": col.replace("_encoded", ""),
      "Negatif (0)": f"{counts[0]} ({counts[0]/tot*100:.1f}%)",
      "Netral (1)": f"{counts[1]} ({counts[1]/tot*100:.1f}%)",
      "Positif (2)": f"{counts[2]} ({counts[2]/tot*100:.1f}%)",
      "Total": tot,
  })
print(pd.DataFrame(rekap).to_string(index=False))

print("\n" + "=" * 68)
print("3. DISTRIBUSI KELAS PER SPLIT (TRAIN / VAL / TEST)")
print("=" * 68)
split_rows = []
for col in encoded_cols:
  c_tr = Y_train[col].value_counts().reindex([0, 1, 2], fill_value=0)
  c_va = Y_val[col].value_counts().reindex([0, 1, 2], fill_value=0)
  c_te = Y_test[col].value_counts().reindex([0, 1, 2], fill_value=0)
  split_rows.append({
      "Aspek": col.replace("_encoded", ""),
      "Train (Neg/Net/Pos)": f"{c_tr[0]} / {c_tr[1]} / {c_tr[2]}",
      "Val (Neg/Net/Pos)": f"{c_va[0]} / {c_va[1]} / {c_va[2]}",
      "Test (Neg/Net/Pos)": f"{c_te[0]} / {c_te[1]} / {c_te[2]}",
  })
print(pd.DataFrame(split_rows).to_string(index=False))

# TF-IDF siap pakai
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

Using Colab cache for faster access to the 'shopee-fashion-reviews-indonesian' dataset.
-> File utama terpilih : 'dataset_labeled_v6.xlsx' (6158 baris)
-> Kolom tersedia      : ['rating', 'ulasan', 'pola_ulasan', 'memiliki_label', 'jumlah_aspek', 'aspek_kualitas', 'aspek_bahan', 'aspek_ukuran', 'aspek_desain', 'aspek_warna', 'aspek_kenyamanan', 'label_json']
-> Kolom teks          : 'ulasan'
-> 6 Kolom aspek       : ['aspek_kualitas', 'aspek_bahan', 'aspek_ukuran', 'aspek_desain', 'aspek_warna', 'aspek_kenyamanan']

1. RINGKASAN JUMLAH BARIS DATA
                      Tahap / Himpunan  Jumlah Baris
Dataset Awal (dataset_labeled_v6.xlsx)          6158
                      Setelah dropna()          6158
           Setelah hapus duplikat teks          6158
          Setelah filter >= 5 karakter          6158
         Final Data Bersih (df_bersih)          6158
                   Data Training (80%)          4926
                 Data Validation (10%)           616
                    Dat